<h1 style="color:#a2516e;font-weight:600;">Generate Text with Recurrent Networks</h1>


> **Generative Networks**<br/>
> RNNs can learn word ordering and make predictions for the next word in a sequence.
>
> **How it works?**<br/>
> We can use RNN units that don't accept an input at each step, but instead take an initial state vector and produce a sequence of outputs.

<h3 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1 | Generative Networks</h3>

For start, we will build a **character-level network** that generates text "letter by letter".

In [27]:
import os

# Set the backend that Keras uses before importing it.
os.environ["KERAS_BACKEND"] = "tensorflow"

# Suppress warnings about GPU support.
# 0=ALL, 1=INFO, 2=WARNING, 3=ERROR
# "2" suppresses info and warnings but errors still show.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import keras
import tensorflow_datasets as tfds
import numpy as np

In [28]:
dataset = tfds.load("ag_news_subset")
ds_train = dataset["train"]
ds_test = dataset["test"]

<h4 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.1 | Building Character Vocabulary</h4>

Split the text into individual characters instead of words. The `TextVectorization` layer cannot do that, so we build a simple character-level tokenizer manually by scanning the training titles.

In [29]:
def extract_text(x):
    return x["title"] + " " + x["description"]


def tupelize(x):
    return (extract_text(x), x["label"])


# Character vocabulary from training titles
chars = set()

for x in ds_train:
    chars.update(x["title"].numpy().decode("utf-8"))

# Create character-to-index and index-to-character mappings.
# Reserve index 0 for padding character.
char_to_idx = {ch: i + 1 for i, ch in enumerate(sorted(chars))}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}


# tf.Tensor(b'AMD Debuts Dual-Core Opteron Processor', shape=(), dtype=string)
# b'AMD Debuts Dual-Core Opteron Processor'
# AMD Debuts Dual-Core Opteron Processor

Add manually to the vocabulary a special token denoting the **end-of-sequence**, `<eos>`.

In [37]:
eos_token = len(char_to_idx) + 1
char_to_idx["<eos>"] = eos_token
idx_to_char[eos_token] = "<eos>"

vocab_size = eos_token + 1

A helper function to encode text into sequences of numbers:

In [42]:
def texts_to_sequences(texts):
    return [[char_to_idx.get(ch, 0) for ch in text] for text in texts]


texts_to_sequences("[Hello, world!]")

[[0],
 [36],
 [61],
 [68],
 [68],
 [71],
 [11],
 [1],
 [79],
 [71],
 [74],
 [68],
 [60],
 [2],
 [0]]

<h4 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.2 | Training a Generative RNN to Generate Titles</h4>

> **How it works?**<br/>
> For each input character in the title, we'll train the network to *generate the next character* as output.<br/>
> This is achieved by forcing each cell of the generative RNN to produce not just the final cell, but also an output, by specifying `return_sequences`.
>
> An input to the network would be a sequence of encoded characters, and the output would be a sequence of the same length, but shifted by one element and terminated by **&lt;eos&gt;**.
>
> ["H", "e", "l", "l", "o"] --> ["e", "l", "l", "o", "&lt;eos&gt;"]

![RNN that generates the next character](./img/image%2003.png)

The function below takes a whole mini-batch of titles as input and converts it into one-hot encoded inputs and next-token targets. The steps are:

1. Using a list comprehension, create a list containing each byte-string tensor converted into a Python string.
2. Turn the words into token IDs using our tokenizer function.
3. Since the sequences are of different length, we fill them up to make them of equal length. By default, the padding value is 0, and the padding occurs from the beginning of each sequence.
4. The function returns two values:
    - The one-hot encoded (and padded) sequence
    - The same input sequence but, shifted to the left (z[:, 1:]) by one token while<br/>
    appending the **eos_token** to the end.

where:
    t : a title in the mini-batch x


In [55]:
# fmt: off
def title_batch(x):
    x = [t.numpy().decode("utf-8") for t in x]
    z = texts_to_sequences(x)
    z = keras.utils.pad_sequences(z)

    input_one_hot = tf.one_hot(z, vocab_size)
    target_one_hot = tf.one_hot(tf.concat(
    [
        z[:, 1:], 
        tf.constant(eos_token, shape=(len(z), 1))
    ], 
    axis=1), 
    vocab_size,)

    return (input_one_hot, target_one_hot)

def title_batch_fn(x):
    x=x["title"]
    a,b=tf.py_function(title_batch,inp=[x],Tout=(tf.float32,tf.float32))
    a.set_shape([None,None,vocab_size])
    b.set_shape([None,None,vocab_size])
    
    return a,b

In [57]:
model = keras.Sequential(
    [
        keras.Input(shape=(None, vocab_size)),
        keras.layers.Masking(),
        keras.layers.LSTM(128, return_sequences=True),
        keras.layers.Dense(vocab_size, activation="softmax"),
    ]
)

model.summary()
model.compile(loss="categorical_crossentropy")
model.fit(ds_train.batch(8).map(title_batch_fn))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_1 (Masking)             │ (None, None, 84)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 128)      │       109,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 84)       │        10,836 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119,892 (468.33 KB)

 Trainable params: 119,892 (468.33 KB)

 Non-trainable params: 0 (0.00 B)

c:\Dev\Python\JupyterNotebooks\Tutorials\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


15000/15000 ━━━━━━━━━━━━━━━━━━━━ 459s 30ms/step - loss: 1.6376


<h4 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.3 | Generating Output</h4>

Now, use the trained model to generate output. Start by decoding text represented by a sequence of token numbers:

In [58]:
def decode(x):
    return "".join([idx_to_char.get(t, "") for t in x])

In [59]:
def generate(model, size=100, start="Today "):
    chars = texts_to_sequences([start])[0]

    for i in range(size):
        out = model(tf.expand_dims(tf.one_hot(chars, vocab_size), 0))[0][-1]
        nc = tf.argmax(out)

        if nc == eos_token:
            break

        chars.append(nc.numpy())

    return decode(chars)

In [60]:
generate(model)

'Today #39;s Section Search Security To Start (AP)'

<h4 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.4 | Sampling Output During Training</h4>

In [61]:
sampling_callback = keras.callbacks.LambdaCallback(on_epoch_end=lambda batch, logs: print(generate(model)))

model.fit(
    ds_train.batch(8).map(title_batch_fn),
    callbacks=[sampling_callback],
    epochs=3,
)

Epoch 1/3
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.3713Today #39; more to strike to buy for the bank of the strike
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 459s 31ms/step - loss: 1.3713
Epoch 2/3
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 1.2843Today #39;s strike to start for the bank of the straight state
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 466s 31ms/step - loss: 1.2843
Epoch 3/3
14999/15000 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.2392Today #39;s Straight Start Service Security Set to Start (Reuters)
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 440s 29ms/step - loss: 1.2392


<h4 style="font-weight:600;color:white;background:#a2516e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.5 | Soft Text Generation and Temperature</h4>

In [62]:
def generate_soft(model, size=100, start="Today ", temperature=1.0):
    chars = texts_to_sequences([start])[0]

    for i in range(size):
        out = model(tf.expand_dims(tf.one_hot(chars, vocab_size), 0))[0][-1]
        probs = tf.exp(tf.math.log(out) / temperature).numpy().astype(np.float64)
        probs = probs / np.sum(probs)
        nc = np.argmax(np.random.multinomial(1, probs, 1))

        if nc == eos_token:
            break

        chars.append(nc)

    return decode(chars)


words = ["Today ", "On Sunday ", "Moscow, ", "President ","Little red riding hood "]

for i in [0.3,0.8,1.0,1.3,1.8]:
    print(f"\n--- Temperature = {i}")
    for j in range(5):
        print(generate_soft(model,size=300,start=words[j],temperature=i))


--- Temperature = 0.3
Today #39;s to the the brows of surviver to start of the finds
On Sunday Seeks Start to Reach Profit (AP)
Moscow, Down of Start to Partner Straight Service (Reuters)
President to lead for storm captain companies for the store (AFP)
Little red riding hood for the the first partnership

--- Temperature = 0.8
Today #39; prodrcom demance highes issues resigning in New space
On Sunday Cuts Blast With And Investor in 2004
Moscow, Militants More UN Set #39;s Gains Street
President #39;s susplan #39; perichan seeks Memailers run online warns (AFP)
Little red riding hood crianch on Marketless

--- Temperature = 1.0
Today wins -- procence could alpaccining to belive Its Diecisic (AFP)
On Sunday To Geop the Rebel Urginencling Will Year &lt;b&gt;...&lt;/b&gt;
Moscow, ASescins quit acquisies cambands in belidrick warns
President in lifical mak for US" man bettar new three in Valuea
Little red riding hood for Linnopiler brains 100 draftmates systems

--- Temperature = 1.3
Toda

<span style="color:#a2516e;font-weight:600;font-style:italic;font-size:1.1em;text-decoration:underline;">References:</span>

[Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)<br/>
[Character-level Text Generation with LSTM](https://keras.io/examples/generative/lstm_character_level_text_generation/)